# 06 - Spatial Visualization on Histology

## Learning objectives
1. Overlay gene expression and QC metrics on the H&E image.
2. Plot known marker genes, checking existence first.
3. Interpret spot-level spatial expression patterns.

## Concept
Now we *use* the registration. `squidpy`/`scanpy` handle the scale-factor math, so you ask
for a gene and get its expression painted onto the tissue - a functional overlay on the
anatomical image, exactly like a PET/CT fusion view.


In [ ]:
# --- Standard setup: make `utils` importable and seed RNGs ---
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / 'utils').exists():
    ROOT = ROOT.parent  # in case the notebook is opened from a subfolder
sys.path.insert(0, str(ROOT))

from utils import st_helpers as st
st.set_seeds()  # reproducibility (seed = 0)
print('Project root:', st.project_root())


In [ ]:
import scanpy as sc
import squidpy as sq

adata = st.load_adata('adata_qc.h5ad')
adata


### QC metrics over tissue
A quick sanity overlay before genes.

In [ ]:
sq.pl.spatial_scatter(adata, color=['total_counts', 'n_genes_by_counts'], ncols=2, size=1.3)


### Marker genes (existence-checked)
Marker panels are dataset/species specific. For this **mouse brain** we use canonical
neuronal/glial markers; for a tumor dataset you would swap in epithelial/immune/stromal/
proliferation markers. We filter the wishlist through `st.genes_present()` so a missing
gene never crashes the plot.


In [ ]:
# Mouse-brain markers (neurons, oligodendrocytes/myelin, astrocytes, etc.).
brain_markers = ['Snap25', 'Mbp', 'Plp1', 'Gfap', 'Aqp4', 'Olig1',
                 'Slc17a7', 'Gad1', 'Hpca', 'Pcp4']
# Tumor panel (used automatically if you switch to a cancer dataset).
tumor_markers = ['EPCAM', 'KRT8', 'PTPRC', 'CD3D', 'COL1A1', 'PECAM1', 'MKI67']

candidates = brain_markers + tumor_markers
present = st.genes_present(adata, candidates)
present = present[:6]  # plot up to 6 for readability
print('Plotting:', present)


In [ ]:
if present:
    sq.pl.spatial_scatter(adata, color=present, ncols=3, size=1.3, cmap='viridis')
else:
    print('No marker genes from the panel are present; inspect adata.var_names.')


**Expected output:** several tissue maps where color = log-normalized expression. In mouse
brain you should see clear anatomy: e.g. `Mbp`/`Plp1` light up white-matter tracts,
`Snap25` is broadly neuronal, `Hpca`/`Pcp4` highlight specific regions.

### Alternative API: `sc.pl.spatial`
scanpy's plotter works too and lets you tune image transparency.

In [ ]:
if present:
    sc.pl.spatial(adata, color=present[0], img_key='hires', size=1.5, alpha_img=0.6)


## How to read these plots
- Each dot is a **spot** (multi-cell), colored by that gene's log-normalized expression.
- Spatially coherent patches of high expression = a region where that program is active.
- Because a spot mixes cells, a 'high' spot means the gene is abundant *in that
  neighborhood*, not that every cell expresses it.

## Common pitfalls
- Passing a gene name that is not in `var_names` (wrong case/species) -> always pre-filter.
- Comparing colors across panels without noting each has its **own color scale**.
- Over-interpreting single bright spots (could be noise) vs coherent regions.

## Interpretation
Spatial overlays turn the abstract matrix into anatomy you can see. This is the core
exploratory tool of spatial transcriptomics.

## What this means biologically
Marker genes reveal tissue architecture molecularly: myelin tracts, neuronal layers, glial
populations. The expression map and the H&E morphology should tell a consistent story -
the foundation for linking morphology to molecules.

---
**Next:** `07_clustering_and_spatial_domains.ipynb`.
